# NLP Text Preprocessing: Complete Pipeline (Google Colab)

Dataset: **Twitter Sentiment Analysis** (`twitter_training.csv`): tweets about different games/brands labelled
Positive / Negative / Neutral / Irrelevant.

**Steps covered**
1. Load and inspect the data
2. Data cleaning: nulls, duplicates, unused columns
3. Text cleaning step by step (lowercase, HTML, URLs, mentions, hashtags, emojis, contractions, numbers, punctuation, repeated letters, extra spaces)
4. Stopword removal (keeping negations like *not*)
5. Tokenization
6. Stemming and Lemmatization
7. Before/after comparison, word frequency, word clouds, n-grams, POS tagging
8. Train/test split and label encoding
9. Text to numbers: Keras Tokenizer + padding, Bag of Words, TF-IDF
10. Save the cleaned dataset

**Run the cells from top to bottom.**

## 1. Imports and NLTK resources

In [ ]:
import os, re, string, warnings
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk

warnings.filterwarnings("ignore")

for pkg in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
print("Ready")

## 2. Load the dataset
Upload `twitter_training.csv` when asked (skip the upload if the file is already in Colab).

In [ ]:
CSV = "twitter_training.csv"

if not os.path.exists(CSV):
    from google.colab import files
    up = files.upload()
    csvs = [n for n in up if n.lower().endswith(".csv")]
    if len(csvs) == 1 and csvs[0] != CSV:
        os.rename(csvs[0], CSV)

# This file has NO header row, so we give the column names ourselves
data = pd.read_csv(CSV, header=None, names=["tweet_id", "entity", "sentiment", "text"])
print(data.shape)
data.head()

## 3. Inspect the data

In [ ]:
data.info()
print("\nMissing values:\n", data.isnull().sum())
print("\nDuplicate rows:", data.duplicated().sum())
print("\nSentiment labels:\n", data["sentiment"].value_counts())
print("\nNumber of entities:", data["entity"].nunique())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
data["sentiment"].value_counts().plot(kind="bar", ax=ax[0], color="#4c78a8", title="Tweets per sentiment")
data["entity"].value_counts().head(10).plot(kind="bar", ax=ax[1], color="#f58518", title="Top 10 entities")
plt.tight_layout(); plt.show()

## 4. Basic data cleaning
Drop missing text, drop duplicates, and drop columns we do not need for text preprocessing.

In [ ]:
print("Before:", data.shape)
data = data.dropna(subset=["text"])
data = data.drop_duplicates(subset=["entity", "sentiment", "text"])
data = data.drop(columns=["tweet_id"]).reset_index(drop=True)
data["text"] = data["text"].astype(str)
print("After :", data.shape)
data["raw_text"] = data["text"]            # keep the original for comparison later

## 5. Text cleaning, step by step on one example
Each step is a small function, so you can see exactly what it changes.

In [ ]:
def to_lower(t):            return t.lower()
def remove_html(t):         return re.sub(r"<.*?>", " ", t)
def remove_urls(t):         return re.sub(r"(https?://\S+|www\.\S+)", " ", t)
def remove_mentions(t):     return re.sub(r"@\w+", " ", t)
def remove_hash_sign(t):    return re.sub(r"#", "", t)           # keep the word, drop the '#'
def remove_emojis(t):
    return re.sub(r"[\U0001F000-\U0001FAFF\U00002600-\U000027BF\U0000FE00-\U0000FE0F]", " ", t)

CONTRACTIONS = [(r"won't", "will not"), (r"can't", "can not"), (r"n't", " not"),
                (r"'re", " are"), (r"'m", " am"), (r"'ll", " will"),
                (r"'ve", " have"), (r"'d", " would"), (r"'s\b", "")]
def expand_contractions(t):
    t = t.replace("’", "'")
    for pat, rep in CONTRACTIONS:
        t = re.sub(pat, rep, t)
    return t

def remove_numbers(t):      return re.sub(r"\d+", " ", t)
def remove_punctuation(t):  return t.translate(str.maketrans(string.punctuation, " " * len(string.punctuation)))
def reduce_repeats(t):      return re.sub(r"(.)\1{2,}", r"\1\1", t)   # "goooood" -> "good"
def remove_extra_spaces(t): return re.sub(r"\s+", " ", t).strip()

steps = [("lowercase", to_lower), ("remove HTML", remove_html), ("remove URLs", remove_urls),
         ("remove @mentions", remove_mentions), ("remove # sign", remove_hash_sign),
         ("remove emojis", remove_emojis), ("expand contractions", expand_contractions),
         ("remove numbers", remove_numbers), ("remove punctuation", remove_punctuation),
         ("reduce repeated letters", reduce_repeats), ("remove extra spaces", remove_extra_spaces)]

example = "OMG!!! I ❤️ this gameeee 😍 @Borderlands, it's soooo gooood!! Visit https://t.co/xyz #Gaming  I can't stop... 100%   <b>wow</b>"
t = example
print("ORIGINAL:", t, "\n")
for name, fn in steps:
    t = fn(t)
    print(f"{name:<26}->", t)

## 6. Stopwords, tokenization, stemming and lemmatization

In [ ]:
# Stopwords: remove very common words, but KEEP negations because they change sentiment
stop_words = set(stopwords.words("english")) - {"no", "nor", "not", "never"}
print("Stopwords used:", len(stop_words))

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

@lru_cache(maxsize=None)
def stem_word(w):  return stemmer.stem(w)

@lru_cache(maxsize=None)
def lemma_word(w): return lemmatizer.lemmatize(lemmatizer.lemmatize(w, "v"), "n")

sample = "the players were running quickly and the games are better than studies"
print("\nword_tokenize :", word_tokenize(sample))
toks = [w for w in word_tokenize(sample) if w not in stop_words]
print("no stopwords  :", toks)
print("stemming      :", [stem_word(w) for w in toks])
print("lemmatization :", [lemma_word(w) for w in toks])

## 7. Full pipeline function, applied to the whole dataset

In [ ]:
def clean_text(t):
    for _, fn in steps:
        t = fn(t)
    return t

def remove_stopwords(tokens):
    return [w for w in tokens if w not in stop_words and len(w) > 1]

# 1) clean   2) tokenize (text is already clean, so split on spaces)   3) remove stopwords
data["clean_text"] = data["raw_text"].apply(clean_text)
data["tokens"] = data["clean_text"].str.split().apply(remove_stopwords)

# 4) stemming and lemmatization
data["stemmed"] = data["tokens"].apply(lambda ws: [stem_word(w) for w in ws])
data["lemmatized"] = data["tokens"].apply(lambda ws: [lemma_word(w) for w in ws])

# final text = lemmatized tokens joined back into a sentence
data["final_text"] = data["lemmatized"].apply(" ".join)

# drop tweets that became empty after cleaning
empty = (data["final_text"].str.len() == 0).sum()
data = data[data["final_text"].str.len() > 0].reset_index(drop=True)
print("Rows that became empty and were dropped:", empty)
print("Final shape:", data.shape)
data[["raw_text", "final_text"]].head(8)

## 8. Before vs after

In [ ]:
data["len_before"] = data["raw_text"].str.split().str.len()
data["len_after"] = data["tokens"].str.len()

print("Average words per tweet  before:", round(data["len_before"].mean(), 2), " after:", round(data["len_after"].mean(), 2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
ax[0].hist(data["len_before"], bins=40, color="#4c78a8"); ax[0].set_title("Words per tweet: before")
ax[1].hist(data["len_after"], bins=40, color="#54a24b"); ax[1].set_title("Words per tweet: after")
plt.show()

all_words = Counter(w for ws in data["lemmatized"] for w in ws)
print("Vocabulary size (unique words):", len(all_words))
top = pd.DataFrame(all_words.most_common(20), columns=["word", "count"])
top.plot(kind="barh", x="word", y="count", figsize=(6, 5), legend=False, title="Top 20 words").invert_yaxis()
plt.show()

### Word clouds per sentiment

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, s in zip(axes.ravel(), sorted(data["sentiment"].unique())):
    text = " ".join(data.loc[data["sentiment"] == s, "final_text"])
    ax.imshow(WordCloud(width=600, height=350, background_color="white").generate(text))
    ax.set_title(s); ax.axis("off")
plt.tight_layout(); plt.show()

## 9. N-grams and POS tagging

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(texts, n, k=12):
    vec = CountVectorizer(ngram_range=(n, n)).fit(texts)
    counts = np.asarray(vec.transform(texts).sum(axis=0)).ravel()
    idx = counts.argsort()[::-1][:k]
    return pd.DataFrame({"ngram": [vec.get_feature_names_out()[i] for i in idx], "count": counts[idx]})

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
top_ngrams(data["final_text"], 2).plot(kind="barh", x="ngram", y="count", ax=ax[0], legend=False, title="Top bigrams").invert_yaxis()
top_ngrams(data["final_text"], 3).plot(kind="barh", x="ngram", y="count", ax=ax[1], legend=False, title="Top trigrams").invert_yaxis()
plt.tight_layout(); plt.show()

# Part-of-speech tagging on a raw sentence (works best before stopword removal)
print(nltk.pos_tag(word_tokenize("Borderlands is an amazing game and I love playing it")))

## 10. Label encoding and train/test split
We split **before** fitting any vectorizer, so no information from the test set leaks into training.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
data["label"] = le.fit_transform(data["sentiment"])
print(dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    data["final_text"], data["label"], test_size=0.2, stratify=data["label"], random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 11. Text to numbers

### 11a. Keras Tokenizer + padding (for deep learning / LSTM models)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tok = Tokenizer(num_words=20000, oov_token="<OOV>")
tok.fit_on_texts(X_train)                       # fit on TRAIN only

train_seq = tok.texts_to_sequences(X_train)
test_seq = tok.texts_to_sequences(X_test)

MAX_LEN = int(np.percentile([len(s) for s in train_seq], 95))
print("Max length used (95th percentile):", MAX_LEN)

X_train_pad = pad_sequences(train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print("Padded train shape:", X_train_pad.shape)
print("\nText     :", X_train.iloc[0])
print("Sequence :", train_seq[0])
print("Padded   :", X_train_pad[0])

### 11b. Bag of Words and TF-IDF (for classic machine learning models)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)
print("Bag of Words :", X_train_bow.shape)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print("TF-IDF       :", X_train_tfidf.shape)

print("\nSome features:", list(tfidf.get_feature_names_out()[:15]))

### Quick sanity check: does the preprocessed text work for classification?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)
print("Logistic Regression accuracy on test set:", round(accuracy_score(y_test, clf.predict(X_test_tfidf)), 4))

## 12. Save the cleaned dataset

In [ ]:
out = data[["entity", "sentiment", "label", "raw_text", "clean_text", "final_text"]]
out.to_csv("twitter_cleaned.csv", index=False)
print("Saved twitter_cleaned.csv", out.shape)

try:
    from google.colab import files
    files.download("twitter_cleaned.csv")
except Exception:
    pass

## Summary of preprocessing steps
| Step | Why |
|---|---|
| Handle nulls and duplicates | Remove empty and repeated rows |
| Lowercase | "Game" and "game" become the same word |
| Remove HTML, URLs, @mentions, emojis | Not useful as words |
| Keep hashtag word | `#gaming` -> `gaming` |
| Expand contractions | `can't` -> `can not` (keeps the negation) |
| Remove numbers and punctuation | Reduce noise |
| Reduce repeated letters | `soooo` -> `soo` |
| Remove extra spaces | Clean spacing |
| Stopword removal (keep *not*, *no*, *never*) | Remove filler words, keep sentiment-carrying negations |
| Tokenization | Split text into words |
| Stemming / Lemmatization | Reduce words to a base form (`running` -> `run`) |
| Label encoding | Sentiment names -> numbers |
| Tokenizer + padding / BoW / TF-IDF | Turn text into numeric input for models |